# CLEF quickstart

This notebook gives q quick guide to:
1. instantiate a registered CLEF backbone,
2. (optionally) load a local checkpoint in a safe, best-effort way, and
3. run a single random ECG-like sample to obtain a compact representation.

In [ ]:
# 1) Imports and device
import os
import torch
from clef.model_config import ModelConfig

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


In [ ]:
# 2) Minimal config: set model name and checkpoint path (edit these)
MODEL_NAME = 'clef'
# Set CKPT_FOLDER to the folder where you downloaded the checkpoints
CKPT_FOLDER = '../models/clef'
# CKPT_FILE can be a relative path under CKPT_FOLDER
CKPT_FILE = 'clef_small.ckpt'  # set to your file

cfg = {
    'checkpoint_path': CKPT_FOLDER,
    'statekey_file': CKPT_FILE,
    "model_size": "small",
    'lead_config': '1lead',
    'num_classes': 4,
}
print('Config prepared')

Config prepared


In [9]:
# 3) Instantiate backbone
backbone = ModelConfig(name=MODEL_NAME).model_class(cfg).to(DEVICE)
backbone.eval()
print('Backbone created:', getattr(backbone, 'name', type(backbone)))

Backbone created: clef/clef-small.ckpt


In [ ]:
# 4) Checkpoint load
ckpt_loaded = False
if CKPT_FOLDER and CKPT_FILE:
    p = os.path.join(CKPT_FOLDER, CKPT_FILE)
    if os.path.exists(p):
        ckpt = torch.load(p, map_location='cpu', weights_only=True)
        for k in ("state_dict", "model", "ecg_model"):
            if isinstance(ckpt, dict) and k in ckpt:
                backbone.load_state_dict(ckpt[k], strict=False); break
        else:
            try:
                backbone.load_state_dict(ckpt, strict=False)
            except Exception:
                pass

In [13]:
# 5) Run a random ECG-like sample and get a compact representation
channels = 1 if cfg.get('lead_config', '12lead') == '1lead' else 12
seq_len = cfg.get('input_len', 5000)
sample = torch.randn(1, channels, seq_len, device=DEVICE)
print('Sample shape:', tuple(sample.shape))

def get_representation(model, x):
    model.eval()
    with torch.no_grad():
        out = model(x)

    return out
    
rep = get_representation(backbone, sample)
print('Representation shape:', tuple(rep.shape))

Sample shape: (1, 1, 5000)
Representation shape: (1, 256)
